# FarmFederate — Real Data Training Pipeline

Trains **24 models** using real local data:
- **Text**: per-class `text.csv` (cleaned BLIP captions + pvvqa)
- **Images**: per-class `images/` folders (PlantVillage + Beans)

### Models trained
| Type | Models | Count |
|------|--------|-------|
| LLM  | DistilBERT, BERT-tiny, RoBERTa-tiny, ALBERT-tiny, MobileBERT | 5 |
| ViT  | ViT-Base, DeiT-tiny, Swin-tiny, ConvNeXT-tiny, EfficientNet | 5 |
| VLM  | concat, attention, gated, clip, flamingo, blip2, coca, unified_io | 8 |
| Fed/Cent | LLM, ViT, VLM (centralized + federated) | 6 |

### Setup
1. **Runtime → Change runtime type → T4 GPU**
2. Upload your `data/` folder to Google Drive at `MyDrive/FarmFederate/data/`
3. Run all cells

In [ ]:
# ── Cell 1: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

In [ ]:
# ── Cell 2: Clone repo ─────────────────────────────────────────────────────
import os

REPO_URL  = 'https://github.com/Solventerritory/FarmFederate-Advisor.git'
REPO_DIR  = '/content/FarmFederate'
BRANCH    = 'feature/multimodal-work'

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

print("Repo ready at", REPO_DIR)

In [ ]:
# ── Cell 3: Copy data from Drive → Colab ──────────────────────────────────
import shutil, os

DRIVE_DATA = '/content/drive/MyDrive/FarmFederate/data'
LOCAL_DATA = '/content/FarmFederate/data'

if os.path.exists(DRIVE_DATA):
    shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)
    print(f"Data copied: {DRIVE_DATA} → {LOCAL_DATA}")
else:
    print(f"WARNING: {DRIVE_DATA} not found.")
    print("Upload your data/ folder to Drive at MyDrive/FarmFederate/data/")

# Verify
print("\nData summary:")
print(f"{'Class':<15} {'Texts':>8} {'Images':>8}")
print("-" * 33)
for cls in ['water_stress', 'nutrient_def', 'pest_risk', 'disease_risk', 'heat_stress']:
    txt_path = f'{LOCAL_DATA}/{cls}/text.csv'
    img_path = f'{LOCAL_DATA}/{cls}/images'
    txt  = len(open(txt_path).readlines()) - 1 if os.path.exists(txt_path) else 0
    imgs = len([f for f in os.listdir(img_path) if f.endswith(('.jpg','.png'))]) if os.path.exists(img_path) else 0
    print(f"{cls:<15} {txt:>8} {imgs:>8}")

In [ ]:
# ── Cell 4: Install dependencies ──────────────────────────────────────────
!pip install -q torch torchvision transformers scikit-learn pandas numpy matplotlib seaborn tqdm pillow
print("Dependencies installed.")

In [ ]:
# ── Cell 5: GPU check ─────────────────────────────────────────────────────
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# ── Cell 6: Configuration ─────────────────────────────────────────────────
import sys
sys.path.insert(0, '/content/FarmFederate/backend')

from FarmFederate_Colab_Complete import Config
from pathlib import Path

# ── Training config ────────────────────────────────────────────
config = Config(
    epochs                = 15,   # optimal per project settings
    batch_size            = 16,
    max_samples_per_class = 800,  # matches available data per class
    fed_rounds            = 5,
    num_clients           = 3,
    learning_rate         = 1e-4,
)

# Save checkpoints & results to Drive so they survive disconnects
config.checkpoint_dir = Path('/content/drive/MyDrive/FarmFederate/checkpoints')
config.output_dir     = Path('/content/drive/MyDrive/FarmFederate/outputs')
config.plots_dir      = Path('/content/drive/MyDrive/FarmFederate/plots')

for d in [config.checkpoint_dir, config.output_dir, config.plots_dir]:
    d.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(f"  epochs            : {config.epochs}")
print(f"  batch_size        : {config.batch_size}")
print(f"  max_samples/class : {config.max_samples_per_class}")
print(f"  fed_rounds        : {config.fed_rounds}")
print(f"  num_clients       : {config.num_clients}")
print(f"  learning_rate     : {config.learning_rate}")
print(f"  checkpoint_dir    : {config.checkpoint_dir}")

In [ ]:
# ── Cell 7: Run full training ──────────────────────────────────────────────
# Trains: 5 LLM + 5 ViT + 8 VLM + 3 centralized + 3 federated = 24 models
# Checkpoints auto-saved to Drive after each model

from FarmFederate_Colab_Complete import run_training_real_data

results = run_training_real_data(config=config)

In [ ]:
# ── Cell 8: Results summary ────────────────────────────────────────────────
print("=" * 70)
print("FINAL RESULTS SUMMARY")
print("=" * 70)

print(f"\n{'Model':<22} {'F1 Micro':>10} {'F1 Macro':>10} {'Accuracy':>10} {'Params':>12}")
print("-" * 68)

print("\n--- LLM Models (Text) ---")
for name, r in results['llm_models'].items():
    print(f"{name:<22} {r['f1']:>10.4f} {r['f1_macro']:>10.4f} {r['accuracy']:>10.4f} {r['params']:>12,}")

print("\n--- ViT Models (Image) ---")
for name, r in results['vit_models'].items():
    print(f"{name:<22} {r['f1']:>10.4f} {r['f1_macro']:>10.4f} {r['accuracy']:>10.4f} {r['params']:>12,}")

print("\n--- VLM Fusion Models (Text + Image) ---")
for name, r in results['vlm_models'].items():
    print(f"{name:<22} {r['f1']:>10.4f} {r['f1_macro']:>10.4f} {r['accuracy']:>10.4f} {r['params']:>12,}")

print("\n--- Centralized vs Federated ---")
print(f"{'Type':<10} {'Centralized F1':>16} {'Federated F1':>14} {'Diff':>8} {'Winner':>12}")
print("-" * 64)
for mt in ['LLM', 'ViT', 'VLM']:
    c = results['centralized'][mt]['f1']
    f = results['federated'][mt]['f1']
    d = f - c
    w = 'Federated' if d > 0 else ('Centralized' if d < 0 else 'Tie')
    print(f"{mt:<10} {c:>16.4f} {f:>14.4f} {d:>+8.4f} {w:>12}")

# Best overall
all_f1 = {}
for n, r in results['llm_models'].items(): all_f1[f'LLM-{n}'] = r['f1']
for n, r in results['vit_models'].items(): all_f1[f'ViT-{n}'] = r['f1']
for n, r in results['vlm_models'].items(): all_f1[f'VLM-{n}'] = r['f1']
best = max(all_f1, key=all_f1.get)
print(f"\nBest model: {best}  F1={all_f1[best]:.4f}")

In [ ]:
# ── Cell 9: Plot training curves ──────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
titles = ['LLM Models', 'ViT Models', 'VLM Fusion']
model_groups = [results['llm_models'], results['vit_models'], results['vlm_models']]

for ax, title, group in zip(axes, titles, model_groups):
    for name, r in group.items():
        hist = r.get('history', {})
        val_f1 = hist.get('val_f1', [])
        if val_f1:
            ax.plot(val_f1, label=name, marker='o', markersize=3)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Val F1')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/FarmFederate/plots/training_curves.png', dpi=150)
plt.show()
print("Saved: training_curves.png")

In [ ]:
# ── Cell 10: Bar chart — F1 comparison across all models ──────────────────
import matplotlib.pyplot as plt
import numpy as np

names, f1s, colors = [], [], []
for n, r in results['llm_models'].items():
    names.append(n); f1s.append(r['f1']); colors.append('#2196F3')
for n, r in results['vit_models'].items():
    names.append(n); f1s.append(r['f1']); colors.append('#4CAF50')
for n, r in results['vlm_models'].items():
    names.append(n); f1s.append(r['f1']); colors.append('#FF5722')

fig, ax = plt.subplots(figsize=(16, 6))
bars = ax.bar(range(len(names)), f1s, color=colors, edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('F1 Score (Micro)')
ax.set_title('FarmFederate — All Models F1 Comparison (Real Data)', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.axhline(y=np.mean(f1s), color='black', linestyle='--', alpha=0.5, label=f'Mean={np.mean(f1s):.3f}')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bar, val in zip(bars, f1s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=7)

# Legend patches
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#2196F3', label='LLM (Text)'),
    Patch(color='#4CAF50', label='ViT (Image)'),
    Patch(color='#FF5722', label='VLM (Multimodal)'),
    plt.Line2D([0],[0], color='black', linestyle='--', label=f'Mean={np.mean(f1s):.3f}'),
])

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/FarmFederate/plots/f1_comparison_all_models.png', dpi=150)
plt.show()
print("Saved: f1_comparison_all_models.png")

In [ ]:
# ── Cell 11: Centralized vs Federated bar chart ────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

model_types = ['LLM', 'ViT', 'VLM']
cent_f1s = [results['centralized'][mt]['f1'] for mt in model_types]
fed_f1s  = [results['federated'][mt]['f1']   for mt in model_types]

x = np.arange(len(model_types))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - w/2, cent_f1s, w, label='Centralized', color='#1976D2')
b2 = ax.bar(x + w/2, fed_f1s,  w, label='Federated',   color='#388E3C')

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(model_types, fontsize=12)
ax.set_ylabel('F1 Score (Micro)')
ax.set_title('Centralized vs Federated Learning', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/FarmFederate/plots/centralized_vs_federated.png', dpi=150)
plt.show()
print("Saved: centralized_vs_federated.png")

In [ ]:
# ── Cell 12: Save results JSON + zip checkpoints ──────────────────────────
import json, shutil

# Results JSON already saved by run_training_real_data, but save a copy here too
results_path = '/content/drive/MyDrive/FarmFederate/outputs/complete_results.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f"Results saved: {results_path}")

# Zip checkpoints
zip_path = '/content/drive/MyDrive/FarmFederate/farmfederate_checkpoints'
shutil.make_archive(zip_path, 'zip', '/content/drive/MyDrive/FarmFederate/checkpoints')
print(f"Checkpoints zipped: {zip_path}.zip")

print("\nAll done! Files saved to Google Drive:")
print("  outputs/complete_results.json")
print("  plots/training_curves.png")
print("  plots/f1_comparison_all_models.png")
print("  plots/centralized_vs_federated.png")
print("  farmfederate_checkpoints.zip")